# 📱 Deteksi Kecanduan Smartphone – Klasifikasi Biner
## Notebook 1: Pre-processing & Data Cleaning
---
Dataset : User Behavior Dataset  
Target  : `Addicted` → **1 = Kecanduan** (User Behavior Class ≥ 4), **0 = Tidak Kecanduan** (< 4)  
Alasan threshold: Kelas 4 (Tinggi) dan 5 (Sangat Tinggi) secara klinis dikategorikan sebagai perilaku adiktif.

### 1.1 Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('✅ Library berhasil diimport')

### 1.2 Load Dataset

In [ ]:
df = pd.read_csv('user_behavior_dataset.csv')

print(f'Shape dataset  : {df.shape}')
print(f'Jumlah baris   : {df.shape[0]}')
print(f'Jumlah kolom   : {df.shape[1]}')
print(f'\nKolom: {list(df.columns)}')
df.head(10)

### 1.3 Informasi Dataset

In [ ]:
print('=== Info Dataset ===')
df.info()

In [ ]:
print('=== Statistik Deskriptif ===')
df.describe().round(2)

### 1.4 Cek Missing Values

In [ ]:
missing     = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing,
    'Persentase (%)': missing_pct.round(2)
})
print('=== Missing Values per Kolom ===')
print(missing_df)

if missing.sum() == 0:
    print('\n✅ Tidak ada missing values!')
else:
    print(f'\n⚠️  Total missing values: {missing.sum()}')

### 1.5 Cek Duplikasi Data

In [ ]:
duplikat = df.duplicated().sum()
print(f'Jumlah baris duplikat: {duplikat}')

if duplikat > 0:
    df = df.drop_duplicates()
    print(f'✅ {duplikat} baris duplikat dihapus. Shape baru: {df.shape}')
else:
    print('✅ Tidak ada data duplikat!')

### 1.6 Konversi Label → Biner (Kecanduan vs Tidak Kecanduan)

> **Dasar Penetapan Threshold:**  
> Mengacu pada literatur (Wayahdi & Ruziq, 2025) dan DSM-5 criteria for behavioral addiction,  
> kelas 4 (Tinggi) dan kelas 5 (Sangat Tinggi) merepresentasikan perilaku penggunaan smartphone  
> yang **problematik dan adiktif**, sehingga:
> - **1 (Kecanduan)** = User Behavior Class **≥ 4**  
> - **0 (Tidak Kecanduan)** = User Behavior Class **< 4**

In [ ]:
# Simpan label asli sebelum binarisasi
df['Original_Class'] = df['User Behavior Class']

# Konversi ke label biner
# 1 = Kecanduan  : User Behavior Class >= 4 (Tinggi & Sangat Tinggi)
# 0 = Tidak Kecanduan : User Behavior Class < 4
df['Addicted'] = (df['User Behavior Class'] >= 4).astype(int)

print('=== Distribusi Label Biner ===')
binary_dist = df['Addicted'].value_counts()
print(f'  0 (Tidak Kecanduan) : {binary_dist[0]} data ({binary_dist[0]/len(df)*100:.1f}%)')
print(f'  1 (Kecanduan)       : {binary_dist[1]} data ({binary_dist[1]/len(df)*100:.1f}%)')

print('\n=== Detail Mapping Kelas Asli → Biner ===')
mapping_df = df.groupby(['Original_Class', 'Addicted']).size().reset_index(name='Count')
label_map = {1:'Sangat Rendah', 2:'Rendah', 3:'Sedang', 4:'Tinggi', 5:'Sangat Tinggi'}
mapping_df['Keterangan'] = mapping_df['Original_Class'].map(label_map)
mapping_df['Label Biner'] = mapping_df['Addicted'].map({0:'0 – Tidak Kecanduan', 1:'1 – Kecanduan'})
print(mapping_df[['Original_Class','Keterangan','Label Biner','Count']].to_string(index=False))

In [ ]:
# Visualisasi distribusi label biner
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_bin = ['#2ecc71', '#e74c3c']
labels_bin = ['0 – Tidak Kecanduan', '1 – Kecanduan']

# Bar chart
axes[0].bar(labels_bin, binary_dist[[0, 1]].values, color=colors_bin,
            edgecolor='black', linewidth=0.8, width=0.5)
axes[0].set_title('Distribusi Label Biner\n(Kecanduan Smartphone)', fontweight='bold')
axes[0].set_ylabel('Jumlah Data')
axes[0].set_ylim(0, max(binary_dist.values) * 1.15)
for i, v in enumerate(binary_dist[[0, 1]].values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold', fontsize=13)

# Pie chart
axes[1].pie(binary_dist[[0, 1]].values, labels=labels_bin,
            autopct='%1.1f%%', colors=colors_bin, startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Proporsi Label Biner', fontweight='bold')

plt.suptitle('Label Target: Deteksi Kecanduan Smartphone (Biner)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('01_distribusi_label_biner.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualisasi distribusi label biner disimpan.')

### 1.7 Deteksi Outlier (IQR Method)

In [ ]:
num_cols = ['App Usage Time (min/day)', 'Screen On Time (hours/day)',
            'Battery Drain (mAh/day)', 'Number of Apps Installed',
            'Data Usage (MB/day)', 'Age']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

outlier_summary = {}
for i, col in enumerate(num_cols):
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary[col] = n_out

    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops=dict(facecolor='#3498db', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(f'{col}\n(Outlier: {n_out})', fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Nilai')

plt.suptitle('Deteksi Outlier – Boxplot per Fitur Numerik', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('01_outlier_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Ringkasan Outlier (IQR Method) ===')
for col, n in outlier_summary.items():
    status = '⚠️ Ada outlier' if n > 0 else '✅ Bersih'
    print(f'  {col:<40}: {n:>3} outlier  {status}')

### 1.8 Hapus Kolom Tidak Relevan & Encoding

In [ ]:
# Hapus User ID dan label asli (bukan fitur prediktif)
drop_cols = ['User ID', 'User Behavior Class', 'Original_Class']
df_clean = df.drop(columns=drop_cols)

print(f'Kolom setelah drop: {list(df_clean.columns)}')
print(f'Shape             : {df_clean.shape}')

# Label Encoding untuk Gender
df_clean['Gender'] = df_clean['Gender'].map({'Male': 0, 'Female': 1})

# Label Encoding untuk Operating System
df_clean['Operating System'] = df_clean['Operating System'].map({'Android': 0, 'iOS': 1})

# One-Hot Encoding untuk Device Model
df_clean = pd.get_dummies(df_clean, columns=['Device Model'], drop_first=True, dtype=int)

print('\n✅ Encoding selesai')
print(f'Shape setelah encoding: {df_clean.shape}')
print(f'Kolom target (Addicted):')
print(df_clean['Addicted'].value_counts())
df_clean.head(5)

### 1.9 EDA – Distribusi Fitur per Kelas

In [ ]:
key_features = ['App Usage Time (min/day)', 'Screen On Time (hours/day)',
                'Battery Drain (mAh/day)', 'Data Usage (MB/day)']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

colors_bin = ['#2ecc71', '#e74c3c']

for i, col in enumerate(key_features):
    for label, color, name in zip([0, 1], colors_bin, ['Tidak Kecanduan', 'Kecanduan']):
        subset = df_clean[df_clean['Addicted'] == label][col]
        axes[i].hist(subset, bins=30, alpha=0.6, color=color, label=name, edgecolor='white')
    axes[i].set_title(f'Distribusi: {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekuensi')
    axes[i].legend()

plt.suptitle('Distribusi Fitur Utama per Label Biner', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('01_distribusi_fitur_biner.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualisasi EDA disimpan.')

### 1.10 Simpan Data Hasil Cleaning

In [ ]:
df_clean.to_csv('data_cleaned_binary.csv', index=False)

print('✅ Data bersih (biner) disimpan ke: data_cleaned_binary.csv')
print(f'Jumlah fitur (X) : {df_clean.shape[1] - 1}')
print(f'Jumlah sampel    : {df_clean.shape[0]}')
print(f'Distribusi target:')
print(f'  0 (Tidak Kecanduan) : {(df_clean["Addicted"]==0).sum()}')
print(f'  1 (Kecanduan)       : {(df_clean["Addicted"]==1).sum()}')
print(f'\nKolom: {list(df_clean.columns)}')